# Toy H1: rational gap on HelpSteer2 with a preference reward model

**Status:** Draft / scratch. Not part of the formal H1-H4 panel.

This notebook walks through the H1 saturation experiment applied to a preference-alignment task:

- Prompts come from `nvidia/HelpSteer2` (deduplicated)
- A small open-weights model generates K samples per prompt
- A reward model trained on HelpSteer2-style data assigns a scalar utility to each (prompt, sample) pair
- We compute $U^\circ_K$, $\bar{U}_K$, $\hat{\mathcal{R}}_K$ via the existing `src.metrics.rational_gap` module — that module is generic over the utility type, so it handles continuous reward-model scores without modification

**Toy parameters:** $M=10$ prompts, $K=16$ samples per prompt. A full H1-style run would go to $M\ge500$, $K=64$.

**Methodology note — continuous utility:**

Rational gap on a continuous utility is well-defined but the interpretation shifts:

- Binary $U \in \{0, 1\}$ (math, code): $U^\circ_K$ = fraction of prompts with at least one correct sample; gap = fraction-of-utility lost
- Continuous $U \in \mathbb{R}$ (RM score): $U^\circ_K$ = mean of best-of-K reward; gap = utility-units lost to sampling variance

Before promoting this to a formal panel, pick a reward-model normalization (sigmoid squash, win-rate calibration, etc.) so numbers are interpretable across datasets and models.

## 0. Setup

In [ ]:
import os, sys
from pathlib import Path

# Find the repo root so `src.*` imports work.
REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'src' / 'metrics' / 'rational_gap.py').exists():
    if REPO_ROOT.parent == REPO_ROOT:
        raise RuntimeError('cannot find repo root; ensure notebook is inside rational-gap-of-LLM-reasoning/')
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
print(f'repo root: {REPO_ROOT}')

# Reuse the formal-pipeline HF cache + mirror if present (autodl layout).
for env_var, default in (
    ('HF_HOME',     '/root/autodl-tmp/hf_cache'),
    ('HF_ENDPOINT', 'https://hf-mirror.com'),
):
    if env_var not in os.environ and Path(default).exists() if env_var == 'HF_HOME' else True:
        os.environ.setdefault(env_var, default)
print(f"HF_HOME:     {os.environ.get('HF_HOME', '~/.cache/huggingface')}")
print(f"HF_ENDPOINT: {os.environ.get('HF_ENDPOINT', '(unset)')}")

import numpy as np
import matplotlib.pyplot as plt
import torch

# --- toy parameters (edit freely) ---
M         = 10                                                   # number of prompts
K         = 16                                                   # samples per prompt
SEED      = 0
GEN_MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'                          # tiny generator; swap for a larger one if you have GPU room
RM_MODEL  = 'Skywork/Skywork-Reward-Llama-3.1-8B-v0.2'            # HelpSteer2-style 8B reward model

print(f'\ngenerator:    {GEN_MODEL}')
print(f'reward model: {RM_MODEL}')
print(f'\ntoy scale: M={M} prompts, K={K} samples per prompt, seed={SEED}')

## 1. Load HelpSteer2 prompts

`nvidia/HelpSteer2` has ~10k rows but many prompts appear multiple times (each prompt paired with 2–4 different responses + ratings). We dedupe to unique prompts and take the first $M$.

In [ ]:
from datasets import load_dataset

ds = load_dataset('nvidia/HelpSteer2', split='train')
print(f'HelpSteer2 train: {len(ds)} rows')
print(f'columns: {list(ds.features.keys())}')

# Dedupe prompts; preserve original order.
seen = set()
unique_prompts = []
for p in ds['prompt']:
    if p not in seen:
        seen.add(p)
        unique_prompts.append(p)
print(f'unique prompts: {len(unique_prompts)}')

prompts = unique_prompts[:M]
print(f'\ntaking the first M={M}\n')
for i, p in enumerate(prompts[:3]):
    print(f'--- prompt {i} ---')
    print(f'  {p[:200]}{"..." if len(p) > 200 else ""}')
    print()

## 2. Generate $K$ samples per prompt

Using vLLM via the project's `VllmRunner` wrapper. `Qwen2.5-0.5B-Instruct` is intentionally tiny so this is fast; substitute a larger model if you have headroom (and bump `gpu_memory_utilization` accordingly).

**Note:** the `with VllmRunner(...)` block releases GPU memory on exit, so the reward model in section 3 can use the same GPU.

In [ ]:
from src.sampling.vllm_runner import VllmRunner, SamplingConfig
from transformers import AutoTokenizer

gen_tok = AutoTokenizer.from_pretrained(GEN_MODEL)

def format_chat(prompt: str) -> str:
    msgs = [{'role': 'user', 'content': prompt}]
    return gen_tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

formatted = [format_chat(p) for p in prompts]

with VllmRunner(GEN_MODEL, gpu_memory_utilization=0.5, enforce_eager=True) as runner:
    cfg = SamplingConfig(temperature=1.0, top_p=1.0, top_k=-1, max_tokens=512)
    samples = runner.sample(formatted, K=K, seed=SEED, config=cfg)

# `samples` is List[List[str]] of shape (M, K).
print(f'generated {len(samples)} prompts × {len(samples[0])} samples = {len(samples) * len(samples[0])} responses')
print(f'\nfirst sample (prompt 0, k=0):')
print(f'  {samples[0][0][:300]}{"..." if len(samples[0][0]) > 300 else ""}')

## 3. Score samples with the reward model

Each (prompt, sample) pair gets a scalar reward from the RM. This populates the $(M, K)$ utility matrix $U$ that feeds the H1 metrics.

Skywork RM outputs roughly in the $[-5, +5]$ range; positive = high human preference, negative = low. We use raw scalars (no normalization) for this toy run.

In [ ]:
from transformers import AutoModelForSequenceClassification

print(f'loading reward model: {RM_MODEL}')
rm_tok = AutoTokenizer.from_pretrained(RM_MODEL)
rm = AutoModelForSequenceClassification.from_pretrained(
    RM_MODEL,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    num_labels=1,
    trust_remote_code=True,
)
rm.eval()
print(f'loaded on device: {next(rm.parameters()).device}')

@torch.no_grad()
def score(prompt: str, response: str) -> float:
    conv = [
        {'role': 'user', 'content': prompt},
        {'role': 'assistant', 'content': response},
    ]
    ids = rm_tok.apply_chat_template(conv, tokenize=True, return_tensors='pt').to(rm.device)
    return rm(ids).logits[0][0].item()

# Build the (M, K) utility matrix.
utility = np.zeros((M, K), dtype=np.float32)
for i in range(M):
    for k in range(K):
        utility[i, k] = score(prompts[i], samples[i][k])
    print(f'prompt {i:2d}: scored K={K} samples  |  mean={utility[i].mean():+.3f}  max={utility[i].max():+.3f}  spread={utility[i].max()-utility[i].min():.3f}')

print(f'\nutility matrix shape: {utility.shape}')
print(f'overall: mean={utility.mean():+.3f}, std={utility.std():.3f}, min={utility.min():+.3f}, max={utility.max():+.3f}')

## 4. Rational gap via `src.metrics.rational_gap`

`compute_rational_gap` is generic over utility type — it accepts any $(M, K)$ float array. The same code that processes binary verifier outputs handles continuous reward-model scores without modification.

In [ ]:
from src.metrics.rational_gap import (
    compute_rational_gap,
    bootstrap_ci_over_prompts,
)

est = compute_rational_gap(utility)
print(f'U_circ_K  (mean of per-prompt max):   {est.U_circ_K:+.3f}')
print(f'U_bar_K   (overall mean utility):     {est.U_bar_K:+.3f}')
print(f'R_hat_K   (rational gap):             {est.R_hat_K:+.3f}')

ci = bootstrap_ci_over_prompts(
    est.per_prompt_R_hat_K, n_resamples=1000, confidence=0.95, seed=SEED,
)
print(f'\n95% bootstrap CI on R_hat_K: [{ci.ci_low:+.3f}, {ci.ci_high:+.3f}]')
print(f'(M={M} prompts is very small; the CI is correspondingly wide. Scale M up before interpreting.)')

## 5. Saturation curve

Build the curve at $K' \in \{1, 2, 4, 8, \ldots, K\}$ by direct truncation of the utility matrix (no resampling — matches the `no_expectation_framing` rule).

In [ ]:
from src.metrics.rational_gap import U_circ_at_K, U_bar_at_K, R_hat_at_K

Ks = [k for k in (1, 2, 4, 8, 16, 32, 64) if k <= K]
u_circs = [U_circ_at_K(utility, k) for k in Ks]
u_bars  = [U_bar_at_K(utility, k)  for k in Ks]
r_hats  = [R_hat_at_K(utility, k)  for k in Ks]

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(Ks, u_circs, marker='^', color='C0', lw=1.6, label=r'$U^\circ_K$ (reachable)')
ax.plot(Ks, u_bars,  marker='v', color='C3', lw=1.6, label=r'$\bar{U}_K$ (mean)')
ax.plot(Ks, r_hats,  marker='o', color='black', lw=1.6, label=r'$\hat{R}_K$ (gap)')
ax.set_xscale('log', base=2)
ax.set_xticks(Ks)
ax.set_xticklabels([str(k) for k in Ks])
ax.set_xlabel(r'$K$ (samples per prompt)')
ax.set_ylabel('utility (reward-model score)')
ax.set_title(
    f"Toy H1 — {GEN_MODEL.split('/')[-1]} on HelpSteer2 (M={M})\n"
    f"reward model: {RM_MODEL.split('/')[-1]}, seed={SEED}",
    fontsize=10,
)
ax.legend(loc='best', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()

out_pdf = REPO_ROOT / 'drafts' / 'outputs' / 'helpsteer2_h1_toy.pdf'
out_pdf.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(out_pdf)
plt.savefig(out_pdf.with_suffix('.png'), dpi=120)
plt.show()
print(f'\nsaved figure to {out_pdf}')

# Also dump the raw utility matrix in case you want to play with it later.
import pickle
out_pkl = out_pdf.with_suffix('.pkl')
with open(out_pkl, 'wb') as f:
    pickle.dump({
        'M': M, 'K': K, 'seed': SEED,
        'gen_model': GEN_MODEL, 'rm_model': RM_MODEL,
        'prompts': prompts, 'samples': samples,
        'utility': utility,
        'aggregates_at_K_max': {'U_circ_K': est.U_circ_K, 'U_bar_K': est.U_bar_K, 'R_hat_K': est.R_hat_K},
        'bootstrap_ci': {'ci_low': ci.ci_low, 'ci_high': ci.ci_high},
    }, f)
print(f'saved raw artifacts to {out_pkl}')

## Discussion / next steps

**What this validates:**

1. `compute_rational_gap` ingests continuous reward-model utilities without modification
2. The H1 saturation pattern (gap grows with K, then plateaus) should be visible even on tiny M — just with very wide CI
3. The full pipeline (load prompts → vLLM sample → RM score → metrics → plot) runs end-to-end on one GPU

**Scope-up checklist before promoting to a formal panel:**

1. **Pick a reward normalization.** Raw RM scores live in roughly $[-5, +5]$; before cross-cell comparison, decide between:
   - sigmoid: $\sigma(\text{score})$ to $[0, 1]$
   - win-rate vs. a fixed baseline (e.g. the assistant's reference response from HelpSteer2)
   - identity (raw scalar, document the range)
2. **Scale to $M \ge 500$, $K = 64$** to match the formal H1 panel and get tight bootstrap CIs.
3. **Cross-model panel.** The cross-model R̂_K comparison is the H1 headline finding. Repeat with at least Qwen2.5-7B / Tülu-3 / Llama-3.1.
4. **Verifier audit.** Stratified 50 random + 50 load-bearing samples per cell; human spot-check that the RM's preferences are sensible (same audit pattern as the binary verifier audit in the methodology memo).
5. **Promote to formal kit.** Add `src/verification/preference_rm.py` wrapping the RM call, register in `interface.py`, add a dataset entry in `configs/datasets.yaml` (HelpSteer2 prompts), then write `scripts/deployment_exp/run_h1_preference_panel.sh`.

**Caveats specific to RM-as-utility:**

- The RM was trained to discriminate good vs. bad responses; it is NOT calibrated to ground truth. High RM score ≠ correct — it means the response matches what humans rated highly in training.
- For benchmarks with verifiable answers (math, code), the binary verifier $U$ is preferable: it measures what we ultimately care about. RM utility is the right tool when the task is open-ended (helpfulness, honesty, harmlessness).
- The RM is a single deterministic function. Once chosen, our experiment's rational gap depends on whatever biases that specific RM has. Reporting numbers with a single RM should be paired with a sensitivity check (re-run with a second RM and see whether the trends survive).
- Bootstrap CI on $M=10$ is essentially uninformative; do not read into the CI of toy runs.